# Suite SVP — The SystemVerilog Patterns dialect

The patterns a `matches` compares a value with, binding its variables (IEEE 1800 12.6): the dialect
`Expressions.SystemVerilog.Patterns` of the SystemVerilog family, its kinds mbse-programs' patterns and a pattern's
parentheses, its `Term` listing them beside SystemVerilog's Expressions' branches, its holder, what a pattern binds, its
editions, its printing, and its evaluation, which refuses each kind by name (mbse-basis
[CAP-231: the SystemVerilog expression dialect, complete](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-231.md)).

In [ ]:
import functools
import json
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.SystemVerilog.Expressions import Evaluators as SVV, Expressions as SV, Text as SVText
from mbse.Expressions.SystemVerilog.Patterns import Expressions as SVP, Text as SVPText
from mbse.Expressions.SystemVerilog.Types import Expressions as SVT
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()  # the framework, the standard dialects SystemVerilog's types use, and SystemVerilog's family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, SystemVerilog.install,
                SVTypes.install, SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install]:
    store.transaction(install)
store.transaction(SystemVerilog.define)  # SystemVerilog's types, which its evaluation reads
store.transaction(SVPatterns.define)  # the editions the Patterns dialect covers
Term = store.registered("Expressions.Framework.Term")


def held(spec, dialect=SVP.DIALECT):
    """The holder of the term `spec` writes, `dialect`'s own (the Patterns dialect's unless given), written in a
    transaction."""
    with store.transaction():
        holders = functools.reduce(getattr, dialect.path.split("."), store.Expressions)
        return holders.Expression(lambda e: e.root(dialect.spec(spec)))


pattern = lambda spec: F.term(held(spec))  # noqa: E731: a term of the Patterns dialect, held by its holder
sv = lambda spec: F.term(held(spec, SV.DIALECT))  # noqa: E731: a term of SystemVerilog's Expressions, held by its holder
spelled = lambda text: lambda i: i.spelling(text)  # noqa: E731: an identifier, of its spelling
name = lambda text: lambda u: u.nameexpression(lambda n: n.name(lambda i: i.identifier(spelled(text))))  # noqa: E731
variable = lambda text: lambda u: u.variablepattern(lambda v: v.name(spelled(text)))  # noqa: E731: `.text`
wildcard = lambda u: u.wildcardpattern(lambda w: w)  # noqa: E731: `.*`

<a id="SVP-01"></a>
## SVP-01 · The Patterns dialect's union, its kinds' schemas in mbse-programs' shapes, the position that holds it, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
# Its kinds: mbse-programs' patterns, named and shaped as it has them, and a pattern's parentheses, kept as written; its
# Term lists them beside SystemVerilog's Expressions' branches, which a pattern may be (a constant pattern), with the
# Preprocessor's macro usage and invalid syntax; it has no helper of its own, its names Expressions' Identifier.
union = lambda dialect: [b.name for b in store.registered(dialect.union).branches]  # noqa: E731: its Term's branches
assert SVP.TAGS == ("variablepattern", "wildcardpattern", "taggedpattern", "structurepattern", "patternmember",
                    "parenthesizedpattern") and SVP.HELPERS == ()
assert union(SVP.DIALECT) == [*SV.TAGS, "error", "missing", *SVP.TAGS, "macrousage"]  # Expressions' installed first
assert not set(SVP.TAGS) & set(union(SV.DIALECT)) and list(SVP.DIALECT.kinds()) == [*SVP.TAGS, "error", "missing"]
assert SVP.DIALECT.name() == "SystemVerilog Patterns" and SVP.DIALECT.tag == "systemverilog_patterns"
assert SVP.DIALECT.path == "SystemVerilog.Patterns" and SVP.DIALECT.union == "Expressions.SystemVerilog.Patterns.Term"
PatternsTerm = store.registered(SVP.DIALECT.union)
kinds = {b.name: b.schema for b in PatternsTerm.branches}
assert [kinds[tag].name for tag in SVP.TAGS] == [f"Expressions.SystemVerilog.Patterns.{kind}" for kind in (
    "VariablePattern", "WildcardPattern", "TaggedPattern", "StructurePattern", "PatternMember", "ParenthesizedPattern")]
assert {tag: list(kinds[tag].properties) for tag in SVP.TAGS} == {  # each a value object, in mbse-programs' order
    "variablepattern": ["name"], "wildcardpattern": [], "taggedpattern": ["member", "pattern"], "structurepattern": ["items"],
    "patternmember": ["name", "pattern"], "parenthesizedpattern": ["pattern"]} and not any(kinds[t].ref for t in SVP.TAGS)
Identifier = store.registered("Expressions.SystemVerilog.Expressions.Identifier")  # the family's one helper of each
assert all(kinds[tag].properties[p].schema is Identifier for tag, p in [("variablepattern", "name"), ("taggedpattern", "member"),
                                                                       ("patternmember", "name")])
assert all(kinds[tag].properties["pattern"].schema is PatternsTerm for tag in ("taggedpattern", "patternmember", "parenthesizedpattern"))
assert kinds["structurepattern"].properties["items"].schema.element is PatternsTerm  # a pattern, an expression or a member
assert F._parts(kinds["structurepattern"]) == ((), ("items",), ()) and F._parts(kinds["taggedpattern"]) == (("pattern",), (), ("member",))
# The position that holds its Term: a `matches`' pattern; its Term is a branch of the framework's, its dialect a singleton
expressions = {b.name: b.schema for b in store.registered(SV.DIALECT.union).branches}
assert expressions["matchesexpression"].properties["pattern"].schema is PatternsTerm
assert {b.name: b.schema for b in Term.branches}["systemverilog_patterns"] is PatternsTerm
assert store.singleton("Expressions.SystemVerilog.Patterns") is store.Instances.Expressions.SystemVerilog.Patterns
# Each dialect of the family reads its kinds, and it reads theirs, where its Term lists them.
assert SV.DIALECT.kind_named("variablepattern") is SVP.DIALECT.kinds()["variablepattern"]
assert SVP.DIALECT.kind_named("binaryexpression") is SV.DIALECT.kinds()["binaryexpression"]
assert SVP.DIALECT.reads() == (SV.DIALECT, SVT.DIALECT, *SV.DIALECT.reads()[2:])
# Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
# validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
Holder = store.registered("Expressions.SystemVerilog.Patterns.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is PatternsTerm and Holder.validate() == []
every = lambda u: u.structurepattern(lambda s: s.items([  # noqa: E731: '{a: .x, b: tagged Valid (.*), c: 1}
    lambda i: i.patternmember(lambda m: m.name(spelled("a")).pattern(variable("x"))),
    lambda i: i.patternmember(lambda m: m.name(spelled("b")).pattern(lambda p: p.taggedpattern(lambda t: t.member(spelled(
        "Valid")).pattern(lambda q: q.parenthesizedpattern(lambda r: r.pattern(wildcard)))))),
    lambda i: i.patternmember(lambda m: m.name(spelled("c")).pattern(SV.literal(1)))]))
with store.transaction():
    holder = store.Expressions.SystemVerilog.Patterns.Expression(lambda e: e.root(SVP.DIALECT.spec(every)))
    linked = store.Schemas.OfExpression(lambda s: s.name("svp.every").expression(lambda a: a.insert(lambda e: e.expression(holder))))
assert Validators.Validate(store)(Holder, holder) == [] and SVP.DIALECT.validate(holder) == []
assert [F.same(entry.expression, holder) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
assert F.dialect_of(F.term(holder)) == SVP.DIALECT.union and SVText.ToText(holder) == "'{a: .x, b: tagged Valid (.*), c: 1}"
for write, read in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(read(Holder, write(Holder, holder)).root, holder.root)
    assert F.same(read(PatternsTerm, write(PatternsTerm, F.term(holder))), F.term(holder))
# The schema refuses a pattern where a union that lacks it is held: an Expressions holder's root.
with raises(Exception, match="'wildcardpattern' is not a member of a union value"):
    with store.transaction():
        store.Expressions.SystemVerilog.Expressions.Expression(lambda e: e.root(SVP.DIALECT.make(F.Form("wildcardpattern"))))
# Another family's term is refused, as before.
with raises(ValueError, match="a Patterns let is not a SystemVerilog Patterns term"):
    SVP.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1, Patterns.Expressions.variable("a")), Patterns.Expressions.DIALECT)))

<a id="SVP-02"></a>
## SVP-02 · Each kind and its checks: what it requires, its identifiers, a structure pattern's members in order or by name; forms and `make`, and inference

In [ ]:
# Each kind's checks: what it requires, its identifiers' spellings, and mbse-programs' check that a structure pattern's
# members are in order or by name, not both; a tagged pattern's pattern is optional (`tagged None`). A constant pattern
# is the expression it is, its names references the scope must bind.
validate = lambda spec, bound=(): SVP.DIALECT.validate(pattern(spec), bound)  # noqa: E731
member = lambda text, inner: lambda i: i.patternmember(lambda m: m.name(spelled(text)).pattern(inner))  # noqa: E731
for spec, problems in [
    (variable("a"), []), (variable("\\a+b"), []), (wildcard, []),
    (lambda u: u.variablepattern(lambda v: v), ["a variablepattern needs a name"]),
    (variable("1a"), ["'1a' is not an identifier"]), (variable("$a"), ["'$a' is not an identifier"]),
    (lambda u: u.taggedpattern(lambda t: t.member(spelled("None"))), []),  # no pattern: `tagged None`
    (lambda u: u.taggedpattern(lambda t: t.member(spelled("Valid")).pattern(variable("a"))), []),
    (lambda u: u.taggedpattern(lambda t: t.pattern(wildcard)), ["a taggedpattern needs a member"]),
    (lambda u: u.taggedpattern(lambda t: t.member(spelled("9v")).pattern(variable("a"))), ["'9v' is not an identifier"]),
    (lambda u: u.structurepattern(lambda s: s.items([variable("a"), wildcard, SV.literal(1)])), []),  # in order
    (lambda u: u.structurepattern(lambda s: s.items([member("a", variable("x")), member("b", SV.literal(1))])), []),  # by name
    (lambda u: u.structurepattern(lambda s: s.items([member("a", variable("x")), variable("y")])),
     ["a StructurePattern's members are in order or by name, not both"]),
    (lambda u: u.structurepattern(lambda s: s.items([lambda i: i.patternmember(lambda m: m.name(spelled("a")))])),
     ["argument 0: a patternmember needs a pattern"]),
    (lambda u: u.structurepattern(lambda s: s.items([member("2a", variable("a"))])), ["argument 0: '2a' is not an identifier"]),
    (lambda u: u.patternmember(lambda m: m.pattern(wildcard)), ["a patternmember needs a name"]),
    (lambda u: u.parenthesizedpattern(lambda p: p.pattern(variable("a"))), []),
    (lambda u: u.parenthesizedpattern(lambda p: p), ["a parenthesizedpattern needs a pattern"]),
    (lambda u: u.parenthesizedpattern(lambda p: p.pattern(variable("1a"))), ["pattern: '1a' is not an identifier"]),
    (name("x"), ["nameexpression 'x' is not bound"]),  # a constant pattern: an expression
    (lambda u: u.structurepattern(lambda s: s.items([SV.binary("+", name("x"), 1)])), ["argument 0: left: nameexpression 'x' is not bound"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))
assert validate(name("x"), {"x"}) == [] and SVP.problems(F.node("wildcardpattern", F.kind_of(pattern(wildcard))[1])) == []
# Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
# from which make writes it again; a variable's name is the name the framework reads it by.
assert all(SVP.DIALECT.kinds()[tag].role == F.APPLICATION for tag in SVP.TAGS)
tree = pattern(lambda u: u.structurepattern(lambda s: s.items([  # '{a: .x, b: tagged V ('{.*, 1}), c: tagged None}
    member("a", variable("x")), member("b", lambda p: p.taggedpattern(lambda t: t.member(spelled("V")).pattern(
        lambda q: q.parenthesizedpattern(lambda r: r.pattern(lambda w: w.structurepattern(lambda s: s.items([wildcard, SV.literal(1)]))))))),
    member("c", lambda p: p.taggedpattern(lambda t: t.member(spelled("None"))))])))
walked = list(F.walk(tree))
assert {F.kind_of(node)[0] for node in walked} == {*SVP.TAGS, "literal"} - {"variablepattern"} | {"variablepattern"}
for node in walked:
    made = pattern(SVP.DIALECT.make(F.form(node)))
    assert F.same(made, node) and made is not node
assert F.form(pattern(variable("x"))) == F.Form("variablepattern", {"name": F.Part("identifier", {"spelling": "x"}, 0)})
assert F.form(pattern(lambda u: u.taggedpattern(lambda t: t.member(spelled("None"))))).arguments == (None,)  # its empty slot
assert F.name_of(pattern(variable("x"))) == "x" and F.name_of(pattern(wildcard)) is None
assert SVP.DIALECT.infer(tree) is D.Anything and SVP.DIALECT.infer(pattern(variable("x"))) is D.Anything
assert SVP.DIALECT.infer(pattern(SV.binary("+", name("x"), 1)), {"x": D.Anything}) is D.Anything

<a id="SVP-03"></a>
## SVP-03 · What a pattern binds: its variables, wherever they nest, which a `matches` binds in its conditional's consequence and the conditions after it in a `&&&`

In [ ]:
# What a pattern binds (`bound_names`): its variables, wherever its tagged, structure, member and parenthesized patterns
# nest them; a wildcard, a constant pattern (an expression, whose names are references), invalid syntax, and a variable
# with no name or no spelling bind none.
binds = lambda spec: SVP.bound_names(pattern(spec))  # noqa: E731
assert binds(variable("a")) == {"a"} and binds(wildcard) == frozenset() and binds(name("a")) == frozenset()
assert binds(lambda u: u.taggedpattern(lambda t: t.member(spelled("None")))) == frozenset()
assert binds(lambda u: u.taggedpattern(lambda t: t.member(spelled("V")).pattern(variable("a")))) == {"a"}
assert binds(lambda u: u.parenthesizedpattern(lambda p: p.pattern(variable("a")))) == {"a"}
assert binds(lambda u: u.parenthesizedpattern(lambda p: p)) == frozenset()
assert binds(lambda u: u.patternmember(lambda m: m.name(spelled("m")).pattern(variable("a")))) == {"a"}  # not its member's name
assert binds(lambda u: u.structurepattern(lambda s: s.items([variable("a"), lambda i: i.structurepattern(lambda t: t.items([
    variable("b"), wildcard])), lambda i: i.taggedpattern(lambda t: t.member(spelled("V")).pattern(variable("c"))), name("d")]))) == {"a", "b", "c"}
assert binds(lambda u: u.structurepattern(lambda s: s.items([]))) == frozenset()
assert binds(lambda u: u.variablepattern(lambda v: v)) == binds(lambda u: u.variablepattern(lambda v: v.name(lambda i: i))) == frozenset()
assert binds(lambda u: u.error(lambda e: e.text(".a +").terms([SVP.DIALECT.spec(variable("a"))]))) == frozenset()  # what invalid syntax holds
# A pattern binds, but is no binder of its own: none of its kinds refines Let, and none has scopes; the `matches` that
# holds it binds its variables in its conditional's consequence and the conditions after it in a `&&&` (12.6.2, 12.6.3),
# not in the value it matches nor in the alternative.
assert not any(F.is_binder(kinds[tag]) or SVP.DIALECT.kinds()[tag].scopes is not None for tag in SVP.TAGS)
matches = lambda value, inner: lambda u: u.matchesexpression(lambda m: m.value(value).pattern(inner))  # noqa: E731
pair = lambda u: u.structurepattern(lambda s: s.items([variable("a"), variable("b")]))  # noqa: E731: '{.a, .b}
chosen = sv(lambda u: u.conditionalexpression(lambda c: c.condition(matches(name("v"), pair)).consequence(
    SV.binary("+", name("a"), name("b"))).alternative(name("a"))))
assert SY.free(chosen, SV.DIALECT) == {"v", "a"}  # the alternative's `a` is free
assert SV.DIALECT.validate(chosen, {"v"}) == ["alternative: nameexpression 'a' is not bound"]
assert SV.DIALECT.kinds()["conditionalexpression"].scopes(F.kind_of(chosen)[1]) == [frozenset(), {"a", "b"}, frozenset()]
assert SY.free(sv(matches(name("a"), variable("a"))), SV.DIALECT) == {"a"}  # the value it matches is outside it
both = sv(lambda u: u.conditionalexpression(lambda c: c.condition(lambda p: p.predicateexpression(lambda q: q.conditions([
    matches(name("v"), variable("a")), SV.binary(">", name("a"), 0),
    matches(name("w"), lambda t: t.taggedpattern(lambda g: g.member(spelled("T")).pattern(variable("b")))),
    SV.binary("<", name("b"), name("a"))]))).consequence(SV.binary("+", name("a"), name("b"))).alternative(SV.literal(0))))
assert SY.free(both, SV.DIALECT) == {"v", "w"} and SV.DIALECT.validate(both, {"v", "w"}) == []
predicate = F.form(both).arguments[0]
assert SV.DIALECT.kinds()["predicateexpression"].scopes(F.kind_of(predicate)[1]) == [
    frozenset(), {"a"}, {"a"}, {"a", "b"}]  # each condition: what those before it matched

<a id="SVP-04"></a>
## SVP-04 · A tree mixing patterns with expressions and types, validated and printed from the Patterns dialect's holder and from an Expressions holder through a `matches`

In [ ]:
# A tree mixing patterns with expressions and types: a pattern's constant (an expression, a cast of a Types term) nested
# in a structure pattern, validated and printed from the Patterns dialect's holder, and from an Expressions holder
# through a `matches`, the position that holds the Patterns dialect's Term.
mixed = lambda u: u.structurepattern(lambda s: s.items([  # noqa: E731
    member("a", variable("x")),
    member("b", lambda p: p.taggedpattern(lambda t: t.member(spelled("Valid")).pattern(
        lambda q: q.structurepattern(lambda r: r.items([variable("y"), wildcard]))))),
    member("c", SV.cast(lambda t: t.integeratomtype(lambda a: a.keyword("int")), SV.binary("+", name("w"), 1))),
    member("d", lambda p: p.parenthesizedpattern(lambda q: q.pattern(variable("z"))))]))
SOURCE = "'{a: .x, b: tagged Valid '{.y, .*}, c: int'(w + 1), d: (.z)}"
own = pattern(mixed)
assert SVPText.ToText(own) == SVText.ToText(own) == SOURCE and SVPText.ToText is SVText.ToText
assert SVP.DIALECT.validate(own) == ["argument 2: pattern: value: left: nameexpression 'w' is not bound"]
assert SVP.DIALECT.validate(own, {"w"}) == [] and SY.free(own, SVP.DIALECT) == {"w"} and SVP.bound_names(own) == {"x", "y", "z"}
cast = F.form(F.form(own).arguments[2]).arguments[0]  # c's pattern: a cast, its type a Types term
assert F.dialect_of(cast) == SVP.DIALECT.union and F.dialect_of(F.form(cast).arguments[0]) == SVT.DIALECT.union
chosen = sv(lambda u: u.conditionalexpression(lambda c: c.condition(lambda m: m.matchesexpression(lambda q: q.value(name("v")).pattern(mixed)))
    .consequence(SV.binary("+", SV.binary("+", name("x"), name("y")), name("z"))).alternative(name("w"))))
assert SVText.ToText(chosen) == f"v matches {SOURCE} ? x + y + z : w"
assert SV.DIALECT.validate(chosen, {"v", "w"}) == [] and SY.free(chosen, SV.DIALECT) == {"v", "w"}
assert SV.DIALECT.validate(chosen) == ["condition: value: nameexpression 'v' is not bound",
                                       "condition: pattern: argument 2: pattern: value: left: nameexpression 'w' is not bound",
                                       "alternative: nameexpression 'w' is not bound"]
assert F.dialect_of(F.form(F.form(chosen).arguments[0]).arguments[1]) == SVP.DIALECT.union  # the matches' pattern
assert SV.DIALECT.infer(chosen, {"v": D.Anything, "w": D.Anything}) is D.Anything
assert SVText.ToText(sv(lambda u: u.predicateexpression(lambda p: p.conditions([
    lambda c: c.matchesexpression(lambda m: m.value(name("v")).pattern(variable("a"))), SV.binary(">", name("a"), 0)])))) == "v matches .a &&& a > 0"

<a id="SVP-05"></a>
## SVP-05 · Invalid syntax where a pattern stands: validated, printed as written, and refused by evaluation

In [ ]:
# Invalid syntax: the Patterns dialect's Term holds `error` and `missing`, where a pattern stands; validation reports
# them, printers write them as written, and evaluation refuses them, as every dialect's.
broken = pattern(lambda u: u.structurepattern(lambda s: s.items([
    lambda i: i.error(lambda e: e.text(".a +").terms([SVP.DIALECT.spec(variable("a"))])), lambda i: i.missing(lambda m: m.kind("pattern"))])))
assert SVP.DIALECT.validate(broken) == ["argument 0: invalid syntax '.a +'", "argument 1: missing pattern"]
assert SVText.ToText(broken) == "'{.a +, }" and SVP.bound_names(broken) == frozenset()
assert [SVP.DIALECT.kind(a)[2].role for a in F.form(broken).arguments] == [F.INVALID, F.INVALID]
tagged_missing = pattern(lambda u: u.taggedpattern(lambda t: t.member(spelled("V")).pattern(lambda p: p.missing(lambda m: m))))
assert SVP.DIALECT.validate(tagged_missing) == ["pattern: missing a term"] and SVText.ToText(tagged_missing) == "tagged V "
with raises(ValueError, match="SystemVerilog Patterns error has no text to print"):
    SVText.ToText(pattern(lambda u: u.parenthesizedpattern(lambda p: p.pattern(lambda e: e.error(lambda r: r)))))
with raises(ValueError, match="cannot evaluate an invalid expression: argument 0: invalid syntax '.a +'"):
    SVV.OfAny(broken)

<a id="SVP-06"></a>
## SVP-06 · Its editions as data: IEEE 1800-2005 to 1800-2023, each with every one of its own kinds

In [ ]:
# Its editions, as data: its define covers IEEE 1800's five, owned by their designations, each found by its
# designation or its common name, each with every one of its own kinds (all of them since IEEE 1800-2005, as
# mbse-programs' SINCE has them) and none of the kinds it reads; run again, it writes nothing.
patterns_ = store.Instances.Expressions.SystemVerilog.Patterns
IEEE1800 = tuple(f"IEEE Std 1800-{year}" for year in (2005, 2009, 2012, 2017, 2023))
assert tuple(FS.standards(patterns_)) == IEEE1800 and SVP.SINCE == dict.fromkeys(SVP.TAGS, 3)
assert SV.EDITIONS[3][0] == IEEE1800[0]  # the fourth edition of the family's, its first
assert FS.tags(FS.find(patterns_, "SystemVerilog-2005")) == ("IEEE Std 1800-2005", "SystemVerilog-2005")
assert FS.find(patterns_, "SystemVerilog-2023") is FS.find(patterns_, "IEEE Std 1800-2023") and FS.find(patterns_, "Verilog-2005") is None
assert FS.properties(FS.find(patterns_, "SystemVerilog-2023")) == {"bodies": ("org.ieee",), "number": "1800",
                                                                  "revision": ("year", 2023), "names": ("SystemVerilog-2023",)}
assert all(FS.kinds(edition) == [kinds[tag] for tag in SVP.TAGS] for edition in FS.standards(patterns_).values())
assert all(tuple(FS.of(patterns_, kinds[tag])) == IEEE1800 for tag in SVP.TAGS)
assert FS.of(patterns_, kinds["literal"]) == {} and FS.of(patterns_, store.registered("Expressions.Framework.Error")) == {}
assert store.transaction(SVPatterns.define)[0]["SystemVerilog_2023"] is FS.find(patterns_, "SystemVerilog-2023")
assert tuple(FS.standards(patterns_)) == IEEE1800  # nothing written again

<a id="SVP-07"></a>
## SVP-07 · Printing every kind as mbse-programs writes patterns

In [ ]:
# Printing, as mbse-programs' printer writes patterns (`pattern_text`): a variable `.name`, a wildcard `.*`, a tagged
# pattern `tagged member pattern`, a structure pattern in order or by name, a member `name: pattern`, parentheses as
# written, and a constant pattern as the expression it is, from the Patterns dialect's holder.
cases = [
    (variable("a"), ".a"), (variable("\\a+b"), ".\\a+b"), (wildcard, ".*"),
    (lambda u: u.taggedpattern(lambda t: t.member(spelled("None"))), "tagged None"),
    (lambda u: u.taggedpattern(lambda t: t.member(spelled("Valid")).pattern(variable("a"))), "tagged Valid .a"),
    (lambda u: u.taggedpattern(lambda t: t.member(spelled("T")).pattern(lambda p: p.taggedpattern(lambda q: q.member(spelled("U")).pattern(wildcard)))),
     "tagged T tagged U .*"),
    (lambda u: u.structurepattern(lambda s: s.items([])), "'{}"),
    (lambda u: u.structurepattern(lambda s: s.items([variable("a"), SV.literal(1), SV.binary("+", name("x"), 1)])), "'{.a, 1, x + 1}"),
    (lambda u: u.structurepattern(lambda s: s.items([member("a", variable("x")), member("b", lambda p: p.taggedpattern(
        lambda t: t.member(spelled("V"))))])), "'{a: .x, b: tagged V}"),
    (lambda u: u.patternmember(lambda m: m.name(spelled("a")).pattern(wildcard)), "a: .*"),
    (lambda u: u.parenthesizedpattern(lambda p: p.pattern(lambda q: q.taggedpattern(lambda t: t.member(spelled("V")).pattern(variable("a"))))),
     "(tagged V .a)"),
    (SV.conditional(name("x"), 1, 2), "x ? 1 : 2"),  # a constant pattern, unparenthesized as mbse-programs writes it
]
for spec, source in cases:
    assert SVText.ToText(pattern(spec)) == source, (source, SVText.ToText(pattern(spec)))
assert {F.kind_of(pattern(spec))[0] for spec, _ in cases} >= set(SVP.TAGS)  # every kind
assert SVPText.TEXTS.keys() == set(SVP.TAGS) and SVPText.PRECEDENCE == {} and SVPText.INLINE == {}

<a id="SVP-08"></a>
## SVP-08 · Evaluation refuses each of its kinds by name, and a `matches` before a conditional's branches are sized

In [ ]:
# Evaluation (Q11): SystemVerilog's evaluator evaluates none of its kinds, and refuses each by name; the `matches`
# that holds them is refused by name too.
refused = [
    (variable("a"), "variablepattern"), (wildcard, "wildcardpattern"),
    (lambda u: u.taggedpattern(lambda t: t.member(spelled("None"))), "taggedpattern"),
    (lambda u: u.structurepattern(lambda s: s.items([wildcard])), "structurepattern"),
    (lambda u: u.patternmember(lambda m: m.name(spelled("a")).pattern(wildcard)), "patternmember"),
    (lambda u: u.parenthesizedpattern(lambda p: p.pattern(wildcard)), "parenthesizedpattern"),
]
assert [tag for _, tag in refused] == list(SVP.TAGS)
for spec, tag in refused:
    with raises(NotImplementedError, match=f"SystemVerilog Patterns {tag} is not evaluated"):
        SVV.OfAny(held(spec))
with raises(NotImplementedError, match="SystemVerilog matchesexpression is not evaluated"):
    SVV.OfAny(held(lambda u: u.matchesexpression(lambda m: m.value(name("v")).pattern(variable("a"))), SV.DIALECT), {"v": 1})
# A conditional's condition is sized first, so its `matches` is refused by name before its consequence, which names what
# the pattern binds, is read: `v matches .a ? a : 0`, not "a" unbound.
v_matches = lambda u: u.matchesexpression(lambda m: m.value(name("v")).pattern(variable("a")))  # noqa: E731
with raises(NotImplementedError, match="SystemVerilog matchesexpression is not evaluated"):
    SVV.OfAny(held(SV.conditional(v_matches, name("a"), 0), SV.DIALECT), {"v": 1})

<a id="SVP-09"></a>
## SVP-09 · Complete against slang: each of slang's pattern syntax kinds is one of its kinds

In [ ]:
# Complete against slang: each of slang's syntax kinds that is a pattern (12.6) maps to one of its kinds, and each of
# its kinds is mapped from one (completeness data, Q9); an ordered member and an expression pattern are the patterns
# and expressions they hold.
grammar = json.loads(Path("../../grammars/systemverilog-slang.json").read_text(encoding="utf-8"))
mapped = {node["node"]: item.partition(".")[2] for node in grammar["nodes"] for item in node.get("kinds", []) + node.get("helpers", [])
          if item.startswith("Patterns.")}
assert mapped == {"NamedStructurePatternMember": "patternmember", "ParenthesizedPattern": "parenthesizedpattern",
                  "StructurePattern": "structurepattern", "TaggedPattern": "taggedpattern", "VariablePattern": "variablepattern",
                  "WildcardPattern": "wildcardpattern"}
assert set(mapped.values()) == set(SVP.TAGS) | set(SVP.HELPERS)
assert all("not" in node for node in grammar["nodes"] if node["node"] in ("OrderedStructurePatternMember", "ExpressionPattern"))